# Chapter 14 — Multi-Agent: When, How and When Not To

*The decision tree starts with: don't, unless.*

## Objective

Build a tiny supervisor with two workers. Each worker has its own `GovernanceHarness` and therefore its own audit chain. Then list the questions to answer before reaching for multi-agent.

In [ ]:
from glassloop.core import BaseAgent, Finish, TaskSpec
from glassloop.governance import GovernanceHarness
from glassloop.multiagent import AgentMessage, MessageBus, Supervisor, Worker
from glassloop.tools import GovernedToolExecutor, ToolRegistry

## Build two workers, each with its own harness

In [ ]:
class FixedFinish(BaseAgent):
    def __init__(self, answer): self.answer = answer
    def propose_action(self, state):
        return Finish(output=self.answer)

def make_worker(name, capability, answer):
    executor = GovernedToolExecutor(ToolRegistry(), gates=[])
    harness = GovernanceHarness(FixedFinish(answer), executor)
    return Worker(name=name, capability=capability, harness=harness)

classifier = make_worker('classifier', 'classify_complaint', 'complaint')
drafter    = make_worker('drafter',    'draft_response',     {'text': 'thank you for reaching out'})

## A supervisor routes by name

Messages are typed `AgentMessage` objects on a shared `MessageBus`. Free-form chatter is the failure mode this module is designed to prevent.

In [ ]:
bus = MessageBus()
sup = Supervisor(name='sup', workers=[classifier, drafter], bus=bus)

r1 = sup.delegate('classifier', goal='classify this message')
r2 = sup.delegate('drafter',    goal='draft a response')
print('classifier response:', r1.payload)
print('drafter response:   ', r2.payload)
print(f'bus has {len(bus)} messages')

## Each worker has an independent audit chain

This is the governance-per-worker guarantee. If `classifier` is later replaced, its history is preserved separately.

In [ ]:
print('classifier audit verifies:', classifier.harness.audit.verify())
print('drafter audit verifies:   ', drafter.harness.audit.verify())
print('classifier chain head[:8]:', classifier.harness.audit.head()[:8])
print('drafter chain head[:8]:   ', drafter.harness.audit.head()[:8])

## When *not* to use multi-agent

Multi-agent is often where systems go wrong. Before reaching for it, answer all three:

1. Can a single well-engineered agent with planning + tools do this?
2. Do the workers really have different capabilities, policies or audit boundaries?
3. Are you prepared for the extra latency, the extra failure surface and the harder evaluation?

If the answer to (1) is yes, stop. If (2) is no, stop. Multi-agent buys you parallel specialization and clean governance boundaries. It does **not** buy you intelligence.

## Anti-patterns flagged here

- Reaching for multi-agent before single-agent is exhausted.
- Free-form agent chatter.
- Voting as a substitute for evidence.

In [ ]:
# Self-check
assert r1.payload['final_output'] == 'complaint'
assert classifier.harness.audit is not drafter.harness.audit
print('OK')